In [1]:
import numpy as np
from numba import cuda

In [ ]:
@cuda.jit
def reduction_global(arr):
    tid = cuda.threadIdx.x

    arr[tid] += arr[tid + 1024]
    cuda.syncthreads()

    stride = 512
    while stride > 0:
        if tid < stride:
            arr[tid] += arr[tid + stride]
        cuda.syncthreads()
        stride //= 2

In [5]:
def run_reduction():
    n = 2048
    arr = np.random.randint(0, 100, n).astype(np.int32)

    d_arr = cuda.to_device(arr)

    reduction_global[1, 1024](d_arr)

    result = d_arr.copy_to_host()

    print("GPU Reduction Result:", result[0])
    print("CPU Check:", np.sum(arr))

In [6]:

if __name__ == "__main__":
    run_reduction()

GPU Reduction Result: 101347
CPU Check: 101347
